# Airbnb listings Silver normalization (F09.1)

Normalizes the Toronto Inside Airbnb detailed-listings Bronze product
(`airbnb_listings_bronze`) into its first Silver product
(`airbnb_listings_silver`), exactly as specified by the versioned
contract `contracts/airbnb/silver_listings.yml`.

Since F09.1 the complete transformation lives in
`src/airbnb/listings_silver.py`: contract-driven, deterministic
normalization of every documented Bronze column into typed Silver
columns (money/percent/boolean/date/timestamp parsing through
null-safe `try_cast` - a malformed value becomes null plus an entry in
`normalization_issues`, never a failed run), the city registry currency
attached without conversion, and full lineage retained so every row
traces back to its landed source file and Bronze run. Every input row is
preserved: no business filtering, no deduplication, no accepted/rejected
split (those belong to F09.2/F09.3).

The Bronze slice is selected by landed identity: the latest successful
Bronze run recorded in the unified ingestion manifest, or an explicit
`source_file_id` (legacy source-manifest rows carry no dataset tag, so
dataset-based landed resolution cannot see the real slices). The write
replaces exactly one city/snapshot slice - the same boundary Bronze
uses.

This notebook supplies only the run widgets and the result display.

In [0]:
# Run-mode controls (F02.2 / F05.3 widget conventions, applied to the
# Silver flow in F09.1).
#
# run_mode defaults to "test": every table this notebook touches is
# resolved into the dedicated trippilot_test catalog through
# src.common.environment. "live" resolves the development tables and is
# used only after the test-mode evidence gate has passed.
#
# city_id must be a registered, enabled city in the city registry; the
# registry also supplies the slice currency (toronto -> CAD, attached
# verbatim, never converted).
#
# source_file_id is OPTIONAL here: blank selects the latest successful
# Bronze run for the city/dataset from the unified ingestion manifest;
# a concrete id pins the exact Bronze slice to reprocess (the Bronze
# notebook convention, needed because the shared family/entity/version
# identity is ambiguous across Toronto files).
dbutils.widgets.dropdown("run_mode", "test", ["test", "live"])
dbutils.widgets.text("city_id", "toronto")
dbutils.widgets.text("source_file_id", "")

RUN_MODE = dbutils.widgets.get("run_mode")
CITY_ID = dbutils.widgets.get("city_id").strip()
SOURCE_FILE_ID = dbutils.widgets.get("source_file_id").strip() or None

print(f"Run mode: {RUN_MODE} | city: {CITY_ID}")
print(f"source_file_id: {SOURCE_FILE_ID or '(auto: latest successful Bronze run)'}")

In [0]:
# Locate the repository root, then run the shared flow (F09.1).
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [start, *start.parents]
        if (p / "src").is_dir() and (p / "config").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate the TripPilot repository root (src/ and config/)."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.airbnb.listings_silver import run_listings_silver_transform

# The flow resolves every destination through the environment resolver,
# selects the Bronze slice by landed identity (ingestion manifest or
# the explicit widget override), verifies the landed record, applies
# the contract normalization and atomically replaces the one
# city/snapshot Silver slice. Rerunning the same slice replaces it with
# identical data - the output is deterministic.
summary = run_listings_silver_transform(
    spark,
    run_mode=RUN_MODE,
    city_id=CITY_ID,
    source_file_id=SOURCE_FILE_ID,
)

In [0]:
# Result display: F09.1 acceptance evidence (normalized sample, any
# conversion issues, Bronze row-quality mix, slice identity).
from pyspark.sql import functions as F

display(summary["silver_df"].limit(5))

issues_df = summary["silver_df"].where(F.size(F.col("normalization_issues")) > 0)
if issues_df.limit(1).count() > 0:
    display(
        issues_df.select(
            "listing_id",
            "normalization_issues",
            "bronze_quality_status",
            "source_file_id",
        ).limit(20)
    )
else:
    print("No conversion issues in this slice.")

display(summary["silver_df"].groupBy("bronze_quality_status").count())

print(
    f"Slice: {summary['city_id']} / {summary['snapshot_date']} / "
    f"{summary['source_file_id']}"
)
print(f"Rows: {summary['bronze_rows']} Bronze -> {summary['silver_rows']} Silver")
print(f"Currency: {summary['currency']}")